In [0]:
%pip install torch torch-geometric
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import torch

from torch_geometric.data import Data
from sklearn.preprocessing import LabelEncoder

In [0]:
clientes = spark.table("clientes_features")
edges = spark.table("edges_raw")

In [0]:
pdf_clientes = clientes.toPandas()
pdf_edges = edges.toPandas()

In [0]:
features = [
    "idade",
    "renda_mensal",
    "score_credito",
    "qtd_produtos",
    "valor_fraude",
    "tempo_resolucao_dias",
    "stress_cliente",
    "pix_por_renda"
]

X = pdf_clientes[features].fillna(0).values

In [0]:
y = pdf_clientes["judicializou"].values

In [0]:
all_nodes = pd.concat([
    pdf_edges["source"],
    pdf_edges["target"]
]).unique()

node_map = {node:i for i,node in enumerate(all_nodes)}

In [0]:
pdf_edges["source_id"] = pdf_edges["source"].map(node_map)
pdf_edges["target_id"] = pdf_edges["target"].map(node_map)

edge_index = torch.tensor(
    pdf_edges[["source_id","target_id"]].values.T,
    dtype=torch.long
)

In [0]:
x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(y, dtype=torch.long)

In [0]:
data = Data(x=x, edge_index=edge_index, y=y)

print(data)